# 02 — ベクトル、単位、座標系

四足制御で最も多い不具合は、値そのものよりshape・脚順・座標系の取り違えです。

**前提**: `01_environment_and_source_map.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 3つの座標系

- World \(W\): 重力と地面に固定
- Base \(B\): 胴体と一緒に回転
- Heading \(H\): yawだけ胴体に追従し、roll/pitchは除く

回転行列を \(R_{A\leftarrow B}\) と書くと、

\[
{}^A v = R_{A\leftarrow B}\,{}^B v,\qquad
R_{B\leftarrow A}=R_{A\leftarrow B}^{T}
\]

位置は原点の差も必要です。速度ベクトルと位置ベクトルを同じ式で変換してはいけません。

In [2]:
import numpy as np
yaw = np.deg2rad(30)
R_W2H = np.array([[np.cos(yaw), np.sin(yaw)],
                  [-np.sin(yaw), np.cos(yaw)]])
v_W = np.array([1.0, 0.0])
v_H = R_W2H @ v_W
print("v_W:", v_W, "m/s")
print("v_H:", np.round(v_H, 3), "m/s")
print("round trip:", np.round(R_W2H.T @ v_H, 12))
assert np.allclose(R_W2H.T @ R_W2H, np.eye(2))

v_W: [1. 0.] m/s
v_H: [ 0.866 -0.5  ] m/s
round trip: [1. 0.]


## shape契約

標準SRBDでは、基本状態は24要素
`(COM位置3, 速度3, Euler角3, 角速度3, 足位置12)`。
実装は積分状態6要素も確保し、実際の `states_dim` は30です。
入力は `(足速度12, 床反力12)` の24要素です。
脚順は `FL, FR, RL, RR` です。

In [3]:
LEG_ORDER = ("FL", "FR", "RL", "RR")
blocks = {"position": 3, "velocity": 3, "rpy": 3, "omega": 3, "feet": 12, "integrals": 6}
print("state dimension:", sum(blocks.values()))
print("input dimension:", 12 + 12)
assert sum(blocks.values()) == 30

state dimension: 30
input dimension: 24


**注意**: 上流 `forward_dynamics` のdocstringには29次元と残っていますが、
コードで連結される状態は30次元です。説明とコードが食い違う場合は、
シンボルの `size()` と実行結果を正にします。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。